# Day 3 — UKF Forecasting & Uncertainty Quantification

Builds on Day 2 MLE parameter recovery.
- **Step 1** — Monte Carlo forecast from UKF posterior (90 % CI, 26-week horizon)
- **Step 2** — Parametric bootstrap 95 % CIs for β, γ, R₀
- **Step 3** — Scenario analysis across four β levels
- **Step 4** — 4-panel diagnostic plot

In [ ]:
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# DAY 1 INFRASTRUCTURE  (copied verbatim)
# =============================================================================

def sir_transition(x, beta, gamma, dt=1.0):
    S, I = x[0], x[1]
    S_next = S - beta * S * I * dt
    I_next = I + (beta * S * I - gamma * I) * dt
    return np.array([S_next, I_next])


def sigma_points(x, P, alpha=1e-3, beta_p=2.0, kappa=0.0):
    n   = len(x)
    lam = alpha**2 * (n + kappa) - n
    c   = n + lam
    Wm    = np.full(2*n + 1, 0.5 / c)
    Wm[0] = lam / c
    Wc    = Wm.copy()
    Wc[0] += (1.0 - alpha**2 + beta_p)
    try:
        L = np.linalg.cholesky(c * P)
    except np.linalg.LinAlgError:
        L = np.linalg.cholesky(c * P + 1e-10 * np.eye(n))
    sigmas        = np.empty((2*n + 1, n))
    sigmas[0]     = x
    for i in range(n):
        sigmas[i + 1]     = x + L[:, i]
        sigmas[n + i + 1] = x - L[:, i]
    return sigmas, Wm, Wc


class UKF1D:
    def __init__(self, beta, gamma, Q, R_obs, x0, P0, dt=1.0,
                 alpha=1e-3, beta_ukf=2.0, kappa=0.0):
        self.beta     = beta
        self.gamma    = gamma
        self.Q        = np.asarray(Q, dtype=float)
        self.R_obs    = float(R_obs)
        self.x        = np.asarray(x0, dtype=float).copy()
        self.P        = np.asarray(P0, dtype=float).copy()
        self.dt       = dt
        self.alpha    = alpha
        self.beta_ukf = beta_ukf
        self.kappa    = kappa

    def _f(self, x):
        return sir_transition(x, self.beta, self.gamma, self.dt)

    def _h(self, x):
        return x[1]

    def predict(self):
        sp, Wm, Wc = sigma_points(self.x, self.P, self.alpha, self.beta_ukf, self.kappa)
        sp_f   = np.array([self._f(s) for s in sp])
        x_pred = Wm @ sp_f
        diff   = sp_f - x_pred
        P_pred = np.einsum('i,ij,ik->jk', Wc, diff, diff) + self.Q
        self.x, self.P = x_pred, P_pred

    def update(self, y_obs):
        sp, Wm, Wc = sigma_points(self.x, self.P, self.alpha, self.beta_ukf, self.kappa)
        sp_h   = np.array([self._h(s) for s in sp])
        y_pred = Wm @ sp_h
        diff_y = sp_h - y_pred
        diff_x = sp   - self.x
        S_yy = np.dot(Wc, diff_y**2) + self.R_obs
        P_xy = np.einsum('i,ij,i->j', Wc, diff_x, diff_y)
        K    = P_xy / S_yy
        nu   = y_obs - y_pred
        self.x = self.x + K * nu
        self.P = self.P - np.outer(K, K) * S_yy
        return nu, S_yy

    def step(self, y_obs):
        self.predict()
        return self.update(y_obs)

    def run(self, observations):
        T           = len(observations)
        states      = np.empty((T, 2))
        innovations = np.empty(T)
        inn_vars    = np.empty(T)
        for t in range(T):
            nu, S_yy        = self.step(observations[t])
            states[t]       = self.x
            innovations[t]  = nu
            inn_vars[t]     = S_yy
        return states, innovations, inn_vars


def ukf_negloglik(params, observations, dt=1.0):
    log_b, log_g, log_s2S, log_s2I, log_s2e, logit_I0 = params
    beta        = np.exp(log_b)
    gamma       = np.exp(log_g)
    sigma2_S    = np.exp(log_s2S)
    sigma2_I    = np.exp(log_s2I)
    sigma2_eps1 = np.exp(log_s2e)
    I0          = 1.0 / (1.0 + np.exp(-logit_I0))
    S0          = 1.0 - I0
    Q   = np.diag([sigma2_S, sigma2_I])
    P0  = np.diag([1e-6, 1e-6])
    x0  = np.array([S0, I0])
    ukf = UKF1D(beta, gamma, Q, sigma2_eps1, x0, P0, dt)
    log_lik = 0.0
    for y in observations:
        ukf.predict()
        nu, S_yy = ukf.update(y)
        if S_yy <= 0 or not np.isfinite(S_yy):
            return 1e12
        log_lik += -0.5 * (np.log(2 * np.pi * S_yy) + nu**2 / S_yy)
    return -log_lik


def estimate_parameters(observations, dt=1.0, n_restarts=5, verbose=True):
    obs    = np.asarray(observations, dtype=float)
    bounds = [
        (-5.0,  1.0),
        (-5.0,  1.0),
        (-14.0, -2.0),
        (-14.0, -2.0),
        (-10.0,  2.0),
        (-8.0,  -0.5),
    ]
    x0_default = np.array([
        np.log(0.30), np.log(0.10),
        np.log(1e-5), np.log(1e-5),
        np.log(5e-3),
        np.log(0.01 / 0.99),
    ])
    best_val, best_result = np.inf, None
    rng_opt = np.random.default_rng(seed=0)
    for i in range(n_restarts):
        x0 = x0_default if i == 0 else np.array(
            [rng_opt.uniform(lo, hi) for lo, hi in bounds]
        )
        try:
            res = minimize(
                ukf_negloglik, x0,
                args=(obs, dt),
                method='L-BFGS-B',
                bounds=bounds,
                options={'maxiter': 2000, 'ftol': 1e-10, 'gtol': 1e-7},
            )
            if verbose:
                print(f'  restart {i}: negloglik = {res.fun:.4f}  success={res.success}')
            if res.fun < best_val:
                best_val, best_result = res.fun, res
        except Exception as e:
            if verbose:
                print(f'  restart {i}: FAILED ({e})')
    p  = best_result.x
    I0 = 1.0 / (1.0 + np.exp(-p[5]))
    best_params = {
        'beta':        np.exp(p[0]),
        'gamma':       np.exp(p[1]),
        'R0':          np.exp(p[0]) / np.exp(p[1]),
        'sigma2_S':    np.exp(p[2]),
        'sigma2_I':    np.exp(p[3]),
        'sigma2_eps1': np.exp(p[4]),
        'I0':          I0,
        'S0':          1.0 - I0,
        'negloglik':   best_val,
    }
    return best_params, best_result

print('Day 1 infrastructure loaded.')

In [ ]:
# =============================================================================
# DAY 2  (copied verbatim — simulation + MLE + re-filter)
# =============================================================================

rng = np.random.default_rng(seed=42)

BETA_TRUE   = 0.30
GAMMA_TRUE  = 0.10
S2_S_TRUE   = 1e-5
S2_I_TRUE   = 1e-5
S2_EPS_TRUE = 5e-3
I0_TRUE     = 0.01
T           = 52

print('True parameters')
print(f'  beta={BETA_TRUE}  gamma={GAMMA_TRUE}  R0={BETA_TRUE/GAMMA_TRUE:.1f}  I0={I0_TRUE}  s2_eps={S2_EPS_TRUE}')

# Step 1 — simulate
S_true = np.empty(T)
I_true = np.empty(T)
S, I = 1.0 - I0_TRUE, I0_TRUE
for t in range(T):
    S_true[t] = S
    I_true[t] = I
    wS = rng.normal(0, np.sqrt(S2_S_TRUE))
    wI = rng.normal(0, np.sqrt(S2_I_TRUE))
    S_next = S - BETA_TRUE * S * I + wS
    I_next = I + (BETA_TRUE * S * I - GAMMA_TRUE * I) + wI
    S = np.clip(S_next, 0.0, 1.0)
    I = np.clip(I_next, 0.0, 1.0 - S)

GT_obs = np.clip(I_true + rng.normal(0, np.sqrt(S2_EPS_TRUE), T), 0.0, 1.0)
weeks  = np.arange(T)

# Step 2 — sanity check
ukf_true = UKF1D(
    beta=BETA_TRUE, gamma=GAMMA_TRUE,
    Q=np.diag([S2_S_TRUE, S2_I_TRUE]),
    R_obs=S2_EPS_TRUE,
    x0=np.array([1.0 - I0_TRUE, I0_TRUE]),
    P0=np.diag([1e-6, 1e-6]),
)
states_true, innov_true, innvar_true = ukf_true.run(GT_obs)
std_innov_true = innov_true / np.sqrt(innvar_true)
print(f'\nFilter (true params) — innov mean: {innov_true.mean():+.5f}  std(v/√S): {std_innov_true.std():.4f}')

# Step 3 — MLE
print('\nRunning MLE (L-BFGS-B, 5 restarts) ...\n')
est_params, opt_result = estimate_parameters(GT_obs, dt=1.0, n_restarts=5, verbose=True)

print('\n=== Recovery ===')
print(f"{'Parameter':<12} {'True':>8} {'Estimated':>10} {'Rel. err':>9}")
print('-' * 44)
for name, true_v, est_v in [
    ('beta',    BETA_TRUE,            est_params['beta']),
    ('gamma',   GAMMA_TRUE,           est_params['gamma']),
    ('R0',      BETA_TRUE/GAMMA_TRUE, est_params['R0']),
    ('I0',      I0_TRUE,              est_params['I0']),
    ('s2_eps1', S2_EPS_TRUE,          est_params['sigma2_eps1']),
]:
    print(f"  {name:<10} {true_v:>8.5f} {est_v:>10.5f} {abs(est_v-true_v)/true_v*100:>8.1f}%")

# Step 4 — re-filter with estimated params
ukf_est = UKF1D(
    beta=est_params['beta'],
    gamma=est_params['gamma'],
    Q=np.diag([est_params['sigma2_S'], est_params['sigma2_I']]),
    R_obs=est_params['sigma2_eps1'],
    x0=np.array([est_params['S0'], est_params['I0']]),
    P0=np.diag([1e-6, 1e-6]),
)
states_est, innov_est, innvar_est = ukf_est.run(GT_obs)
std_innov_est = innov_est / np.sqrt(innvar_est)
print(f'\nFilter (est params) — innov mean: {innov_est.mean():+.5f}  std(v/√S): {std_innov_est.std():.4f}')

---
## Day 3 — Forecasting & Uncertainty Quantification

In [ ]:
# =============================================================================
# DAY 3  Step 1 — Monte Carlo Forecast from UKF Posterior
# =============================================================================

rng3    = np.random.default_rng(seed=99)
N_AHEAD = 26     # forecast horizon (weeks)
N_PATHS = 500    # Monte Carlo sample paths

print('=' * 60)
print('DAY 3 — FORECASTING & UNCERTAINTY QUANTIFICATION')
print('=' * 60)
print(f'\nStep 1: Monte Carlo forecast  ({N_PATHS} paths, {N_AHEAD} weeks ahead) ...')

# Re-run filter to get final posterior mean x_post and covariance P_post
ukf_fc = UKF1D(
    beta=est_params['beta'],
    gamma=est_params['gamma'],
    Q=np.diag([est_params['sigma2_S'], est_params['sigma2_I']]),
    R_obs=est_params['sigma2_eps1'],
    x0=np.array([est_params['S0'], est_params['I0']]),
    P0=np.diag([1e-6, 1e-6]),
)
ukf_fc.run(GT_obs)
x_post = ukf_fc.x.copy()   # posterior mean  [S_T, I_T]
P_post = ukf_fc.P.copy()   # posterior covariance

# Sample N_PATHS initial forecast states from N(x_post, P_post)
L_post    = np.linalg.cholesky(P_post + 1e-10 * np.eye(2))
x_samples = x_post + (L_post @ rng3.standard_normal((2, N_PATHS))).T
x_samples = np.clip(x_samples, 0.0, 1.0)

# Propagate each path forward with process noise
Q_fc = np.diag([est_params['sigma2_S'], est_params['sigma2_I']])
L_Q  = np.linalg.cholesky(Q_fc)

I_fc = np.empty((N_PATHS, N_AHEAD))
S_fc = np.empty((N_PATHS, N_AHEAD))

for p_idx in range(N_PATHS):
    x = x_samples[p_idx].copy()
    for step in range(N_AHEAD):
        x = sir_transition(x, est_params['beta'], est_params['gamma'])
        x = np.clip(x + L_Q @ rng3.standard_normal(2), 0.0, 1.0)
        I_fc[p_idx, step] = x[1]
        S_fc[p_idx, step] = x[0]

fc_weeks = np.arange(T, T + N_AHEAD)
I_fc_med = np.median(I_fc, axis=0)
I_fc_lo  = np.percentile(I_fc,  5, axis=0)
I_fc_hi  = np.percentile(I_fc, 95, axis=0)

print(f'  Posterior mean  : S={x_post[0]:.4f}  I={x_post[1]:.4f}')
print(f'  Median peak I   : {I_fc_med.max():.4f}  at forecast week {I_fc_med.argmax() + 1}')
print(f'  90% CI at peak  : [{I_fc_lo[I_fc_med.argmax()]:.4f}, {I_fc_hi[I_fc_med.argmax()]:.4f}]')

In [ ]:
# =============================================================================
# DAY 3  Step 2 — Parametric Bootstrap 95 % CIs
# =============================================================================

B_BOOT = 100    # bootstrap resamples  (increase to 500 for publication)
print(f'\nStep 2: Parametric bootstrap  (B={B_BOOT}, 2 restarts each) ...')
print('  This takes a few minutes — grab a coffee.')

boot_beta, boot_gamma, boot_R0 = [], [], []

for b in range(B_BOOT):
    # Simulate new dataset from estimated model
    S_b, I_b = est_params['S0'], est_params['I0']
    I_sim    = np.empty(T)
    for t in range(T):
        I_sim[t] = I_b
        wS    = rng3.normal(0, np.sqrt(est_params['sigma2_S']))
        wI    = rng3.normal(0, np.sqrt(est_params['sigma2_I']))
        S_new = np.clip(S_b - est_params['beta'] * S_b * I_b + wS, 0.0, 1.0)
        I_new = np.clip(
            I_b + (est_params['beta'] * S_b * I_b - est_params['gamma'] * I_b) + wI,
            0.0, 1.0 - S_new
        )
        S_b, I_b = S_new, I_new

    obs_b = np.clip(
        I_sim + rng3.normal(0, np.sqrt(est_params['sigma2_eps1']), T),
        0.0, 1.0
    )

    try:
        ep_b, _ = estimate_parameters(obs_b, dt=1.0, n_restarts=2, verbose=False)
        boot_beta.append(ep_b['beta'])
        boot_gamma.append(ep_b['gamma'])
        boot_R0.append(ep_b['R0'])
    except Exception:
        pass

    if (b + 1) % 25 == 0:
        print(f'  bootstrap {b+1}/{B_BOOT}  (collected {len(boot_beta)} valid)')

boot_beta  = np.array(boot_beta)
boot_gamma = np.array(boot_gamma)
boot_R0    = np.array(boot_R0)

print(f'\n  Valid resamples: {len(boot_beta)}/{B_BOOT}')
print(f"\n  {'Param':<8} {'CI 2.5%':>9} {'CI 97.5%':>10} {'True':>8} {'Covers?':>9}")
print('  ' + '-' * 48)
for name, arr, true_v in [
    ('beta',  boot_beta,  BETA_TRUE),
    ('gamma', boot_gamma, GAMMA_TRUE),
    ('R0',    boot_R0,    BETA_TRUE / GAMMA_TRUE),
]:
    lo, hi   = np.percentile(arr, [2.5, 97.5])
    covers   = 'YES' if lo <= true_v <= hi else 'NO'
    print(f'  {name:<8} {lo:>9.4f} {hi:>10.4f} {true_v:>8.4f} {covers:>9}')

In [ ]:
# =============================================================================
# DAY 3  Step 3 — Scenario Analysis (four β levels)
# =============================================================================

print('\nStep 3: Scenario analysis')

SCENARIOS = {
    'Low  (β×0.6)':    est_params['beta'] * 0.6,
    'Base (β×1.0)':    est_params['beta'] * 1.0,
    'High (β×1.4)':    est_params['beta'] * 1.4,
    'Severe (β×2.0)':  est_params['beta'] * 2.0,
}
all_weeks  = np.arange(T + N_AHEAD)
scenario_I = {}

for label, beta_s in SCENARIOS.items():
    S_s, I_s = 1.0 - I0_TRUE, I0_TRUE
    I_path   = np.empty(T + N_AHEAD)
    for t in range(T + N_AHEAD):
        I_path[t] = I_s
        S_n = np.clip(S_s - beta_s * S_s * I_s, 0.0, 1.0)
        I_n = np.clip(
            I_s + (beta_s * S_s * I_s - est_params['gamma'] * I_s),
            0.0, 1.0 - S_n
        )
        S_s, I_s = S_n, I_n
    scenario_I[label] = I_path
    print(f'  {label}  R0={beta_s/est_params["gamma"]:.2f}  peak I={I_path.max():.4f}  at week {I_path.argmax()}')

In [ ]:
# =============================================================================
# DAY 3  Step 4 — 4-panel Diagnostic Plot
# =============================================================================

SCEN_COLORS = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    'Day 3 — Forecasting & Uncertainty Quantification\n'
    f'Est: β={est_params["beta"]:.3f},  γ={est_params["gamma"]:.3f},  '
    f'R\u2080={est_params["R0"]:.2f}',
    fontsize=12, fontweight='bold'
)

# --- Panel 1: Forecast with 90 % CI -----------------------------------------
ax = axes[0, 0]
ax.scatter(weeks, GT_obs, s=14, c='grey', alpha=0.5, zorder=1, label='GT obs (in-sample)')
ax.plot(weeks, states_est[:, 1], 'k-', lw=1.8, label='UKF filter (est. params)')
ax.fill_between(fc_weeks, I_fc_lo, I_fc_hi, alpha=0.25, color='tomato', label='90 % CI')
ax.plot(fc_weeks, I_fc_med, 'r--', lw=2.0, label='Median forecast')
ax.axvline(T - 0.5, color='navy', ls=':', lw=1.2, label='Forecast start')
ax.set(xlabel='Week', ylabel='I (normalised)',
       title=f'UKF Forecast  ({N_PATHS} paths, 90 % CI)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# --- Panel 2: Scenario Analysis ---------------------------------------------
ax = axes[0, 1]
ax.scatter(weeks, GT_obs, s=14, c='grey', alpha=0.4, zorder=1, label='GT obs')
for (label, I_path), color in zip(scenario_I.items(), SCEN_COLORS):
    ax.plot(all_weeks, I_path, color=color, lw=2.0, label=label)
ax.axvline(T - 0.5, color='navy', ls=':', lw=1.2)
ax.set(xlabel='Week', ylabel='I (normalised)', title='Scenario Analysis  (4 β levels)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# --- Panel 3: Bootstrap histograms for β and γ ------------------------------
ax = axes[1, 0]
for arr, true_v, color, name in [
    (boot_beta,  BETA_TRUE,  '#2196F3', 'β'),
    (boot_gamma, GAMMA_TRUE, '#4CAF50', 'γ'),
]:
    lo, hi = np.percentile(arr, [2.5, 97.5])
    ax.hist(arr, bins=25, alpha=0.55, color=color,
            label=f'{name}  95 % CI [{lo:.3f}, {hi:.3f}]')
    ax.axvline(true_v, color=color, ls='--', lw=2.0)
ax.set(xlabel='Parameter value', ylabel='Count',
       title=f'Bootstrap CIs for β, γ  (B={len(boot_beta)})\n(dashed = true value)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# --- Panel 4: R0 bootstrap distribution ------------------------------------
ax = axes[1, 1]
lo_r0, hi_r0 = np.percentile(boot_R0, [2.5, 97.5])
ax.hist(boot_R0, bins=30, color='#9C27B0', alpha=0.7,
        label=f'R\u2080  95 % CI [{lo_r0:.2f}, {hi_r0:.2f}]')
ax.axvline(BETA_TRUE / GAMMA_TRUE, color='black', ls='--', lw=2.0,
           label=f'True R\u2080 = {BETA_TRUE/GAMMA_TRUE:.1f}')
ax.axvline(est_params['R0'], color='red', ls=':', lw=2.0,
           label=f'Est. R\u2080 = {est_params["R0"]:.2f}')
ax.set(xlabel='R\u2080', ylabel='Count',
       title=f'Bootstrap Distribution of R\u2080  (B={len(boot_R0)})')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('day3_forecast_uncertainty.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved -> day3_forecast_uncertainty.png')